# Brain-AI Multimodal Usage Notebook

This notebook demonstrates how to generate synthetic multimodal data, build pipeline combinations, run experiments, compare results, and create a DAG artifact.

In [ ]:
import pandas as pd

from brain_ai.core.brain import Brain
from brain_ai.dag.builder import DAGBuilder
from brain_ai.decision.engine import DecisionEngine, generate_pipeline_combinations
from brain_ai.experiments.evaluator import Evaluator
from brain_ai.utils.datasets import generate_multimodal_regression_data

In [ ]:
data = generate_multimodal_regression_data(n_samples=150, random_state=42)
{name: values.shape for name, values in data['modalities'].items()}

In [ ]:
specs = generate_pipeline_combinations(
    modalities=['tabular', 'sensor', 'text'],
    granularity_strategies=['resample', 'pooling', 'attention'],
    fusion_strategies=['early', 'late', 'intermediate'],
    model_backends=['sklearn'],
    hyperparameters={'n_estimators': 20, 'random_state': 42},
)
len(specs)

In [ ]:
brain = Brain(decision_engine=DecisionEngine(), evaluator=Evaluator())
rows = []
for idx, spec in enumerate(specs):
    result = brain.run_pipeline(spec, data)
    rows.append({
        'spec_idx': idx,
        'granularity': spec.granularity_strategy,
        'fusion': spec.fusion_strategy,
        'model': spec.model_backend,
        'score': result['metrics'].get('score'),
        'rmse': result['metrics'].get('rmse'),
        'mae': result['metrics'].get('mae'),
    })
comparison = pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True)
comparison.head(10)

In [ ]:
best_spec = specs[int(comparison.iloc[0]['spec_idx'])]
dag_builder = DAGBuilder(out_dir='examples')
dag_path = dag_builder.build_and_save(best_spec, filename='multimodal_best_dag.png')
dag_path

## What To Check
1. Fusion and granularity strategies should produce different metrics.
2. All combinations should run successfully.
3. A DAG image should be generated at `examples/multimodal_best_dag.png`.